# Day 4 · 05 · Time Series

Data with a timestamp gets its own toolkit in pandas: a `DatetimeIndex`, resampling, rolling windows and time zones.

**In this notebook**
- Parsing dates and building a `DatetimeIndex`
- Finding gaps in a regular series
- `resample`: change the frequency
- Rolling, expanding and exponentially weighted windows
- `shift`, `diff` and `pct_change`
- Daily and weekly patterns
- Time zones
- Rates over time: ratio of sums
- Spotting spikes with a rolling median
- Periods, offsets and date arithmetic

**Dataset:** `site_traffic.csv`: hourly website sessions, conversions and page-load time for January to March 2025. It has missing hours and missing values on purpose. `orders.csv` and `customers.csv` are used for date arithmetic.

In [1]:
from pathlib import Path

import pandas as pd

DATA = Path("../datasets").resolve()

traffic = pd.read_csv(DATA / "site_traffic.csv", parse_dates=["timestamp"]).set_index("timestamp").sort_index()

print("rows        :", len(traffic))
print("from / to   :", traffic.index.min(), "/", traffic.index.max())
print("sorted      :", traffic.index.is_monotonic_increasing, "| unique:", traffic.index.is_unique)
traffic.head(3)

rows        : 2124
from / to   : 2025-01-01 00:00:00 / 2025-03-31 23:00:00
sorted      : True | unique: True


,sessions,conversions,avg_load_ms
timestamp,,,
2025-01-01 00:00:00,64.0,2.0,446.3
2025-01-01 01:00:00,90.0,4.0,1015.3
2025-01-01 02:00:00,88.0,2.0,425.5


## 1. The `DatetimeIndex`
- `parse_dates` converts the text column to real timestamps while reading.
- With timestamps as a sorted index you can select by partial date strings.
- Components such as `.hour`, `.day_name()` and `.day_of_week` are available on the index. On a column, use the `.dt` accessor.

In [2]:
print("rows on 14 Feb:", len(traffic.loc["2025-02-14"]))

parts = pd.DataFrame(
    {
        "hour": traffic.index.hour,
        "weekday": traffic.index.day_name(),
        "day_of_week": traffic.index.day_of_week,   # Monday = 0
        "week": traffic.index.isocalendar().week.to_numpy(),
        "month": traffic.index.month_name(),
    },
    index=traffic.index,
)
parts.loc["2025-01-05 22:00":"2025-01-06 01:00"]

rows on 14 Feb: 24


,hour,weekday,day_of_week,week,month
timestamp,,,,,
2025-01-05 22:00:00,22,Sunday,6,1,January
2025-01-05 23:00:00,23,Sunday,6,1,January
2025-01-06 00:00:00,0,Monday,0,2,January
2025-01-06 01:00:00,1,Monday,0,2,January


## 2. Is the series complete?
A regular series should have one row per period. Missing **rows** are harder to notice than missing values, because nothing in the data says they are absent.

Build the timestamps you expect and compare.

In [3]:
expected = pd.date_range(traffic.index.min(), traffic.index.max(), freq="h")
missing = expected.difference(traffic.index)

print("expected hours:", len(expected))
print("present       :", len(traffic))
print("missing       :", len(missing))

expected hours: 2160
present       : 2124
missing       : 36


In [4]:
# Group consecutive missing hours into runs to find the outages.
gaps = pd.Series(missing)
run_id = (gaps.diff() != pd.Timedelta(hours=1)).cumsum()

runs = gaps.groupby(run_id).agg(start="min", end="max", hours="size")
runs.sort_values("hours", ascending=False).head(3)

,start,end,hours
19,2025-03-05 01:00:00,2025-03-05 06:00:00,6
21,2025-03-07 03:00:00,2025-03-07 04:00:00,2
3,2025-01-14 19:00:00,2025-01-14 19:00:00,1


`asfreq("h")` inserts the missing timestamps as rows filled with `NaN`. From here on every hour has a row, so positions and windows mean what they say.

In [5]:
hourly = traffic.asfreq("h")

print("rows after asfreq:", len(hourly))
hourly.loc["2025-03-05 00:00":"2025-03-05 07:00"]

rows after asfreq: 2160


,sessions,conversions,avg_load_ms
timestamp,,,
2025-03-05 00:00:00,120.0,3.0,460.6
2025-03-05 01:00:00,NaN,NaN,NaN
2025-03-05 02:00:00,NaN,NaN,NaN
2025-03-05 03:00:00,NaN,NaN,NaN
2025-03-05 04:00:00,NaN,NaN,NaN
2025-03-05 05:00:00,NaN,NaN,NaN
2025-03-05 06:00:00,NaN,NaN,NaN
2025-03-05 07:00:00,106.0,3.0,402.9


## 3. `resample`: change the frequency
`resample` is a `groupby` over time periods. Choose an aggregation that fits each column: totals for counts, means for measurements.

| Alias | Period | | Alias | Period |
| --- | --- | --- | --- | --- |
| `"h"` | hour | | `"W"` | week (ending Sunday) |
| `"D"` | day | | `"ME"` / `"MS"` | month end / month start |

In [6]:
daily = hourly.resample("D").agg({"sessions": "sum", "conversions": "sum", "avg_load_ms": "mean"})
daily["hours_observed"] = hourly["sessions"].resample("D").count()

daily.round(1).head(4)

,sessions,conversions,avg_load_ms,hours_observed
timestamp,,,,
2025-01-01,3057.0,103.0,452.1,22
2025-01-02,3611.0,114.0,415.2,24
2025-01-03,3776.0,117.0,422.7,23
2025-01-04,4350.0,133.0,422.4,24


### Pitfall: totals over incomplete periods
`sum` skips missing values, so a day with missing hours gets a **lower total** that looks like a real drop. On 10 February only 12 of 24 hours were recorded.

Keep a count of observations next to every total, and either flag incomplete periods or use the mean.

In [7]:
view = daily.loc["2025-02-08":"2025-02-12", ["sessions", "hours_observed"]].copy()
view["mean_per_hour"] = hourly["sessions"].resample("D").mean().round(1)
view["complete"] = view["hours_observed"] == 24
view

,sessions,hours_observed,mean_per_hour,complete
timestamp,,,,
2025-02-08,4777.0,24,199.0,True
2025-02-09,4289.0,23,186.5,False
2025-02-10,2157.0,12,179.8,False
2025-02-11,3653.0,24,152.2,True
2025-02-12,3517.0,23,152.9,False


In [8]:
# Other frequencies work the same way.
monthly = hourly.resample("ME").agg(
    sessions=("sessions", "sum"), mean_load_ms=("avg_load_ms", "mean"), hours_observed=("sessions", "count")
)
monthly.round(1)

,sessions,mean_load_ms,hours_observed
timestamp,,,
2025-01-31,115427.0,428.7,717
2025-02-28,110317.0,433.4,632
2025-03-31,133410.0,424.7,705


## 4. Rolling, expanding and exponentially weighted windows
A window function computes a statistic over a sliding set of rows.

| Method | Window |
| --- | --- |
| `rolling(7)` | the last 7 rows |
| `rolling("7D")` | the last 7 days of time, however many rows that is |
| `expanding()` | everything up to the current row |
| `ewm(span=7)` | all history, recent rows weighted more |

`rolling(7)` returns `NaN` until 7 values are available. `min_periods` relaxes that.

In [9]:
daily_mean = hourly["sessions"].resample("D").mean()   # robust to missing hours

trend = pd.DataFrame(
    {
        "daily": daily_mean,
        "rolling_7": daily_mean.rolling(7).mean(),
        "rolling_7_min1": daily_mean.rolling(7, min_periods=1).mean(),
        "expanding": daily_mean.expanding().mean(),
        "ewm_7": daily_mean.ewm(span=7).mean(),
    }
).round(1)
trend.head(8)

,daily,rolling_7,rolling_7_min1,expanding,ewm_7
timestamp,,,,,
2025-01-01,139.0,NaN,139.0,139.0,139.0
2025-01-02,150.5,NaN,144.7,144.7,145.5
2025-01-03,164.2,NaN,151.2,151.2,153.6
2025-01-04,181.2,NaN,158.7,158.7,163.7
2025-01-05,169.0,NaN,160.8,160.8,165.4
2025-01-06,144.9,NaN,158.1,158.1,159.2
2025-01-07,140.6,155.6,155.6,155.6,153.8
2025-01-08,141.6,156.0,156.0,153.9,150.4


The 7-day window removes the weekly pattern, which leaves the underlying trend.

In [10]:
weekly_trend = trend["rolling_7"].resample("W").last()
weekly_trend.iloc[[1, 4, 8, 12]]

timestamp
2025-01-12    157.6
2025-02-02    167.6
2025-03-02    181.5
2025-03-30    192.9
Name: rolling_7, dtype: float64

## 5. `shift`, `diff` and `pct_change`
- `shift(n)` moves values `n` rows down, so each row can see an earlier one.
- `diff(n)` is the value minus the value `n` rows earlier.
- `pct_change(n)` is the relative change.

Comparing with the **same weekday last week** (`n=7`) avoids mistaking the weekly pattern for growth.

In [11]:
change = pd.DataFrame(
    {
        "sessions": daily_mean,
        "previous_day": daily_mean.shift(1),
        "change": daily_mean.diff(),
        "vs_previous_day_pct": daily_mean.pct_change().mul(100),
        "vs_last_week_pct": daily_mean.pct_change(7).mul(100),
    }
).round(1)
change.loc["2025-01-10":"2025-01-14"]

,sessions,previous_day,change,vs_previous_day_pct,vs_last_week_pct
timestamp,,,,,
2025-01-10,162.5,150.6,11.9,7.9,-1.0
2025-01-11,186.7,162.5,24.2,14.9,3.0
2025-01-12,176.0,186.7,-10.8,-5.8,4.1
2025-01-13,151.8,176.0,-24.1,-13.7,4.8
2025-01-14,141.9,151.8,-9.9,-6.5,1.0


## 6. Daily and weekly patterns
Group by a component of the timestamp to see seasonality: the average for each hour of the day, or each weekday.

In [12]:
by_hour = hourly.groupby(hourly.index.hour)["sessions"].mean().round(0)
print("busiest hours :", by_hour.nlargest(3).to_dict())
print("quietest hours:", by_hour.nsmallest(3).to_dict())

days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
hourly.groupby(hourly.index.day_name())["sessions"].mean().reindex(days).round(1)

busiest hours : {20: 274.0, 19: 265.0, 21: 262.0}
quietest hours: {2: 96.0, 3: 96.0, 4: 96.0}


timestamp
Monday       160.7
Tuesday      157.8
Wednesday    159.0
Thursday     162.6
Friday       180.4
Saturday     205.5
Sunday       195.8
Name: sessions, dtype: float64

In [13]:
# Hour-by-weekday grid: the usual input for a heat map.
profile = hourly.assign(hour=hourly.index.hour, weekday=hourly.index.day_name()).pivot_table(
    index="hour", columns="weekday", values="sessions", aggfunc="mean"
)[days]
profile.round(0).iloc[8:14]

weekday,Monday,Tuesday,Wednesday,Thursday,Friday,Saturday,Sunday
hour,,,,,,,
8,109.0,107.0,110.0,113.0,126.0,142.0,135.0
9,133.0,127.0,132.0,144.0,156.0,172.0,162.0
10,164.0,162.0,154.0,170.0,192.0,211.0,204.0
11,196.0,193.0,191.0,197.0,222.0,243.0,236.0
12,201.0,207.0,203.0,206.0,229.0,260.0,261.0
13,198.0,204.0,201.0,210.0,231.0,265.0,255.0


## 7. Time zones
Timestamps without a zone are **naive**. Two steps make them usable across regions:
1. `tz_localize` states which zone the recorded times are in.
2. `tz_convert` shows the same instants in another zone.

Assume the server recorded these timestamps in UTC. For a team in India the daily peak falls at a different clock hour.

In [14]:
utc = hourly.tz_localize("UTC")
india = utc.tz_convert("Asia/Kolkata")

print("first timestamp, UTC  :", utc.index[0])
print("same instant, India   :", india.index[0])
print("peak hour of day, UTC  :", utc.groupby(utc.index.hour)["sessions"].mean().idxmax())
print("peak hour of day, India:", india.groupby(india.index.hour)["sessions"].mean().idxmax())

first timestamp, UTC  : 2025-01-01 00:00:00+00:00
same instant, India   : 2025-01-01 05:30:00+05:30
peak hour of day, UTC  : 20
peak hour of day, India: 1


Store and compute in UTC. Convert to local time only for display and for "hour of day" style analysis.

## 8. Rates over time
A rate for a period is the **ratio of the sums**, not the mean of hourly rates. Averaging hourly rates gives a quiet hour at night the same weight as a busy afternoon.

In this dataset the two columns happen to be close. The gap grows whenever the rate differs between busy and quiet hours.

In [15]:
weekly = hourly.resample("W").agg({"sessions": "sum", "conversions": "sum"})
weekly["rate_pct"] = 100 * weekly["conversions"] / weekly["sessions"]
weekly["mean_of_hourly_rates_pct"] = (100 * hourly["conversions"] / hourly["sessions"]).resample("W").mean()

weekly.round(2).head(5)

,sessions,conversions,rate_pct,mean_of_hourly_rates_pct
timestamp,,,,
2025-01-05,18342.0,582.0,3.17,3.15
2025-01-12,25846.0,785.0,3.04,3.02
2025-01-19,26619.0,847.0,3.18,3.22
2025-01-26,26546.0,892.0,3.36,3.30
2025-02-02,26402.0,847.0,3.21,3.16


## 9. Spotting spikes with a rolling median
The median is barely affected by outliers, so a rolling median gives a stable local baseline. Values far above it are spikes.

In [16]:
load = hourly["avg_load_ms"]
baseline = load.rolling(24, center=True, min_periods=12).median()
spikes = load[load > 2 * baseline]

print("hours with load more than twice the local median:", len(spikes))
pd.DataFrame({"avg_load_ms": spikes, "local_median": baseline.loc[spikes.index].round(1)}).head(4)

hours with load more than twice the local median: 24


,avg_load_ms,local_median
timestamp,,
2025-01-01 01:00:00,1015.3,425.4
2025-01-10 22:00:00,1468.1,423.8
2025-01-12 01:00:00,1222.9,405.6
2025-01-16 03:00:00,1631.7,404.0


## 10. Periods, offsets and date arithmetic
- Subtracting two timestamps gives a `Timedelta`. On a column, `.dt.days` extracts whole days.
- A `Period` represents a whole span such as "March 2025".
- `Timedelta` adds an exact duration. `DateOffset` and `offsets` follow the calendar.

In [17]:
orders = pd.read_csv(DATA / "orders.csv", parse_dates=["order_date"])
customers = pd.read_csv(DATA / "customers.csv", parse_dates=["signup_date"])

first_order = orders.groupby("customer_id")["order_date"].min().rename("first_order")
signup = customers.set_index("customer_id")["signup_date"]

days_to_first_order = (first_order - signup).dt.days.dropna()
print(days_to_first_order.describe().round(0)[["count", "min", "50%", "max"]].to_string())

count     205.0
min        14.0
50%       532.0
max      1127.0


In [18]:
stamp = pd.Timestamp("2025-01-31")

print("+ 30 days          :", (stamp + pd.Timedelta(days=30)).date())
print("+ 1 calendar month :", (stamp + pd.DateOffset(months=1)).date())
print("next month end     :", (stamp + pd.offsets.MonthEnd(1)).date())
print("+ 3 business days  :", (stamp + pd.offsets.BDay(3)).date())
print("as a period        :", stamp.to_period("M"), "| quarter:", stamp.to_period("Q"))

orders["order_date"].dt.to_period("Q").value_counts().sort_index()

+ 30 days          : 2025-03-02
+ 1 calendar month : 2025-02-28
next month end     : 2025-02-28
+ 3 business days  : 2025-02-05
as a period        : 2025-01 | quarter: 2025Q1


order_date
2024Q1    241
2024Q2    300
2024Q3    293
2024Q4    437
2025Q1    324
2025Q2    405
Freq: Q-DEC, Name: count, dtype: int64

## Key takeaways
- Put timestamps in a sorted `DatetimeIndex`.
- Check for missing periods before anything else; `asfreq` makes gaps visible.
- `resample` changes frequency. Keep an observation count beside every total.
- Rolling windows smooth; `shift` and `pct_change` compare with earlier periods.
- Localise to a time zone, then convert.
- Compute rates as a ratio of sums.

## Practice
1. Find the day with the highest total conversions among **complete** days.
2. Compute a 24-hour rolling mean of `avg_load_ms` and find its maximum.
3. Compare average weekend and weekday sessions for each month.
4. Convert the series to `America/New_York` and find the peak hour there.